In [0]:
# =============================================================
# NOTEBOOK:  07_bronze_api_weather
# PURPOSE:   Ingest weather data from api into Bronze Delta
# SOURCE:    weather api
# TARGET:    Bronze Delta Tables (bronze/tables/external_api/weather_hourly_forecast/)
# PROJECT:   RetailMax Lakehouse
# =============================================================

import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import uuid
from delta.tables import DeltaTable
import requests
import time


# ── Load Config ───────────────────────────────────────────────
config_path = "/Workspace/Repos/retailmax-lakehouse/retailmax-lakehouse/configs/dev/config.json"

with open(config_path, 'r') as f:
    config = json.load(f)

# ── Storage Paths ─────────────────────────────────────────────
storage_account = config['storage']['account_name']
bronze_path     = config['storage']['bronze_path']
scope_name      = config['secret_scope']

# ── Retrieve Secrets ──────────────────────────────────────────
client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

# ── Configure Spark for ADLS ──────────────────────────────────
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)



In [0]:
def call_api_with_retry(
    url,
    timeout=30,
    max_retries=3,
    backoff_seconds=2
):
    RETRYABLE_CODES = [429, 500, 502, 503, 504]
    NON_RETRYABLE_CODES = [400, 401, 403, 404]

    for attempt in range(max_retries):
        wait = backoff_seconds * (2 ** attempt)

        try:
            response = requests.get(url, timeout=timeout)
            
            if response.status_code == 200:
                return response.json()
            
            elif response.status_code in NON_RETRYABLE_CODES:
                raise Exception(f"Non-retryable error: {response.status_code}")
            
            elif response.status_code in RETRYABLE_CODES:
                print(f"Attempt {attempt + 1}/{max_retries}: Retryable error {response.status_code}. Waiting {wait}s...")
                time.sleep(wait)
        
        except requests.exceptions.Timeout:
            print(f"Attempt {attempt + 1}/{max_retries}: Timeout. Waiting {wait}s...")
            time.sleep(wait)
        
        except requests.exceptions.ConnectionError:
            print(f"Attempt {attempt + 1}/{max_retries}: Connection error. Waiting {wait}s...")
            time.sleep(wait)

    raise Exception(f"API call failed after {max_retries} attempts")


In [0]:
api_endpoint = 'https://api.open-meteo.com/v1/forecast?latitude=40.71&longitude=-74.01&hourly=temperature_2m,precipitation,windspeed_10m&timezone=America/New_York'

data = call_api_with_retry(api_endpoint)
 
weather_data = [{'api_endpoint':api_endpoint,'latitude':data["latitude"],'longitude':data['longitude'],'timezone':data["timezone"],'timezone_abbreviation':data["timezone_abbreviation"],'elevation':data["elevation"],'forecast_granularity':'hourly','hour_count':len(data["hourly"]["time"]),'raw_response':json.dumps(data)}]
# print(weather_data)

df = spark.createDataFrame(weather_data)
# display(df)

batch_id = uuid.uuid4().hex
df = (df
    .withColumn("_load_timestamp", F.current_timestamp())
    .withColumn("_load_date", F.current_date())
    .withColumn("_batch_id", F.lit(batch_id))
)
# display(df)

df.write\
    .mode('append')\
    .format('delta')\
    .save(bronze_path + "tables/external_api/weather_hourly_forecast/")


In [0]:
df_weather = spark.read.format("delta")\
    .load(bronze_path + "tables/external_api/weather_hourly_forecast/")

print(f"Row count: {df_weather.count()}")
df_weather.printSchema()

In [0]:
# Get sample JSON
sample_json = df_weather.select("raw_response").first()[0]

# Infer schema
inferred_schema = spark.read.json(
    spark.sparkContext.parallelize([sample_json])
).schema

print(inferred_schema)

In [0]:
df_parsed = df_weather.withColumn(
    "parsed",
    F.from_json(F.col("raw_response"), inferred_schema)
)

df_parsed.printSchema()

In [0]:
df_hourly_zipped = df_parsed.withColumn(
    "hourly_zipped",
    F.arrays_zip('parsed.hourly.time','parsed.hourly.temperature_2m','parsed.hourly.precipitation','parsed.hourly.windspeed_10m')
)

df_hourly_zipped.printSchema()

In [0]:
df_hourly_record = df_hourly_zipped.withColumn(
    "hourly_record",
    F.explode('hourly_zipped')
)

print(df_hourly_record.count())
df_hourly_record.printSchema()

In [0]:
df_final = df_hourly_record.select('hourly_record.time','hourly_record.temperature_2m','hourly_record.precipitation','hourly_record.windspeed_10m')

df_final.show(5)

In [0]:
def call_api_with_retry(
    url,
    timeout=30,
    max_retries=3,
    backoff_seconds=2
):
    RETRYABLE_CODES = [429, 500, 502, 503, 504]
    NON_RETRYABLE_CODES = [400, 401, 403, 404]

    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=timeout)
            
            if response.status_code == 200:
                return response.json()   # success, done
            
            elif response.status_code in NON_RETRYABLE_CODES:
                raise Exception(f"Non-retryable error: {response.status_code}")
            
            elif response.status_code in RETRYABLE_CODES:
                wait = backoff_seconds * (2 ** attempt)
                print(f"Retryable error {response.status_code}. Waiting {wait}s...")
                time.sleep(wait)
                # loop continues to next attempt
        
        except requests.exceptions.Timeout:
            wait = backoff_seconds * (2 ** attempt)
            print(f"Timeout. Waiting {wait}s...")
            time.sleep(wait)
        
        except requests.exceptions.ConnectionError:
            wait = backoff_seconds * (2 ** attempt)
            print(f"Connection error. Waiting {wait}s...")
            time.sleep(wait)

    # if we get here, all retries exhausted
    raise Exception(f"API call failed after {max_retries} attempts")
